In [3]:
"""
Model scoringowy - dorzuca do popytu z model_popytu.ipynb kilka rzeczy,
ktorych tamten model nie uwzglednial:

1. Pewnosc danych (traffic_confidence) jako mnoznik dla korytarza, i flaga
   (nie mnoznik) dla zanizonych danych EV w segmencie docelowym - kierunek
   tego bledu jest znany i jednokierunkowy, wiec dodatkowe obnizanie by go
   tylko poglebilo

2. Luka AFIR - czy lokalizacja lezy w miejscu, gdzie regulacyjnie powinien
   powstac hub (rozstaw >=150kW/400kW co 60km na TEN-T, Rozporzadzenie UE
   2023/1804), niezaleznie od tego czy akurat jest tam duzy ruch

3. Czytelne skladowe wyniku - zeby dla kazdej lokalizacji dalo sie latwo
   sprawdzic z czego sklada sie jej wynik

4. Sanity-check na Tesla + GreenWay + ORLEN

5. POPRAWKI INŻYNIERSKIE:
   - Deduplikacja stacji EIPA po identycznych wspolrzednych geograficznych
   - Uzupełnienie zerowych i brakujących mocy EIPA medianą (46.0 kW)
   - Zabezpieczenie przed 0 kWh dla punktow na obrzezach siatki GUS
   - Ujednolicenie przelicznika kWh/sesja w segmencie docelowym (35 kWh/sesje)
   - Wyliczanie rankingu NMS Top 1000 wylacznie dla nowych lokalizacji inwestycyjnych
   - WYGŁADZONA KOREKTA KONKURENCJI (użycie np.sqrt mocy, aby zapobiec drastycznym spadkom scoringu)

Wymaga: ../data/candidate_locations_po_korekcie_gus.csv
Wynik: ../data/candidate_locations_ze_scoringiem.csv
"""

import numpy as np
import pandas as pd

PLIK_WEJSCIOWY = "../data/candidate_locations_po_korekcie_gus.csv"
PLIK_WYJSCIOWY = "../data/candidate_locations_ze_scoringiem.csv"

# Mnozniki i progi
MNOZNIK_PEWNOSCI = {"wysoka": 1.00, "srednia": 0.90, "niska": 0.75}
PROG_MOCY_AFIR_KW = 400

DYSTANS_MIN_KM = 30
DYSTANS_MAX_KM = 60
WAGA_AFIR = 0.20

PRZEBIEG_KM = 18266.5
ZUZYCIE_KWH_100KM = 21.0
PHEV_WSPOLCZYNNIK = 0.30
PHEV_KOREKTA = 1.796

RUCH_MNOZNIK_MIN = 0.85
RUCH_MNOZNIK_MAX = 1.15
DOSTAWCZE_MNOZNIK_MIN = 0.92
DOSTAWCZE_MNOZNIK_MAX = 1.12

WAGA_KLASY_TENT = {
    "bazowa": 1.00,
    "bazowa rozszerzona": 0.85,
    "kompleksowa": 0.65,
}
WAGA_KONKURENCJI_PIERSCIEN = 0.5
MOC_TYPOWEJ_STACJI_KW = 44.0

ALFA_KONKURENCJA = 0.030

SREDNIA_ENERGIA_SESJI_DOCELOWA = 35.0  # kWh / sesja
TOP_N_PUNKTOW = 1000

# ZMIANA 1: Promień NMS zmieniony z 5.0 na 2.0 km
PROMIEN_NMS_KM = 1.0

# ZMIANA 2: Bufor wykluczenia od istniejącej stacji EIPA (100 m = 0.1 km)
BUFOR_WYKLUCZENIA_EIPA_KM = 0.10


def haversine_km(lat1, lon1, lat2, lon2):
    """odleglosc w km miedzy punktem a tablica punktow"""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def oznacz_top_n_nms(df_seg, top_n=1000, radius_km=2.0, score_col="wynik_scoringowy", lat_col="latitude", lon_col="longitude"):
    """Algorytm Non-Maximum Suppression (NMS) do wyznaczania unikalnych lokalizacji"""
    df_sorted = df_seg.sort_values(by=score_col, ascending=False).copy()
    selected_indices = []
    suppressed = set()
    lats = df_sorted[lat_col].values
    lons = df_sorted[lon_col].values
    indices = df_sorted.index.values

    for i in range(len(df_sorted)):
        idx = indices[i]
        if idx in suppressed:
            continue
        selected_indices.append(idx)
        if len(selected_indices) == top_n:
            break
        dists = haversine_km(lats[i], lons[i], lats[i+1:], lons[i+1:])
        close_mask = dists < radius_km
        suppressed.update(indices[i+1:][close_mask])

    return {idx: pos + 1 for pos, idx in enumerate(selected_indices)}


def oblicz_luke_afir(df):
    """dla lokalizacji na TEN-T liczy dystans do najblizszego huba
    spelniajacego prog AFIR i zwraca mnoznik premii"""
    huby = df[(df["source_layer"] == "eipa_station") & (df["total_power_kw"] >= PROG_MOCY_AFIR_KW)]
    huby_lat = huby["latitude"].values
    huby_lon = huby["longitude"].values
    print(f"Huby spelniajace prog AFIR (>={PROG_MOCY_AFIR_KW}kW): {len(huby)}")

    dystans_do_huba = pd.Series(np.nan, index=df.index)
    na_tent = df["ten_t"].notna()

    for idx in df[na_tent].index:
        lat, lon = df.at[idx, "latitude"], df.at[idx, "longitude"]
        odlegli = haversine_km(lat, lon, huby_lat, huby_lon)
        dystans_do_huba.at[idx] = odlegli.min() if len(odlegli) > 0 else np.nan

    udzial_luki = ((dystans_do_huba - DYSTANS_MIN_KM) / (DYSTANS_MAX_KM - DYSTANS_MIN_KM)).clip(0, 1)
    waga_klasy = df["ten_t"].map(WAGA_KLASY_TENT).fillna(0)
    mnoznik = 1 + WAGA_AFIR * udzial_luki.fillna(0) * waga_klasy
    return dystans_do_huba, mnoznik


def main():
    df = pd.read_csv(PLIK_WEJSCIOWY, low_memory=False)
    print(f"Wczytano {len(df)} wierszy z pliku wejsciowego.")

    # Deduplikacja stacji EIPA
    maska_eipa = df["source_layer"] == "eipa_station"
    df_eipa = df[maska_eipa].copy()
    df_reszta = df[~maska_eipa].copy()

    df_eipa_dedup = df_eipa.groupby(["latitude", "longitude"], as_index=False).first()
    moc_max = df_eipa.groupby(["latitude", "longitude"])["total_power_kw"].max().values
    df_eipa_dedup["total_power_kw"] = moc_max

    df = pd.concat([df_reszta, df_eipa_dedup], ignore_index=True)
    print(f"Po deduplikacji stacji EIPA liczba wszystkich wierszy: {len(df)}")

    # Uzupełnienie zerowych mocy EIPA medianą
    maska_eipa_aktualna = df["source_layer"] == "eipa_station"
    mediana_moc_eipa = df.loc[maska_eipa_aktualna & (df["total_power_kw"] > 0), "total_power_kw"].median()
    
    maska_zero = maska_eipa_aktualna & ((df["total_power_kw"] == 0) | df["total_power_kw"].isna())
    df.loc[maska_zero, "total_power_kw"] = mediana_moc_eipa
    print(f"Uzupełniono moc dla {maska_zero.sum()} stacji EIPA wartością mediany: {mediana_moc_eipa} kW")

    # Pewność danych
    df["pewnosc_danych_mnoznik"] = 1.0
    kor = df["segment"] == "korytarzowa"
    df.loc[kor, "pewnosc_danych_mnoznik"] = df.loc[kor, "traffic_confidence"].map(MNOZNIK_PEWNOSCI).fillna(1.0)

    df["dane_prawdopodobnie_zanizone"] = df["powiat_ev_dane_zanizone"].fillna(False)

    print("Licze odleglosc do najblizszego hubu AFIR...")
    dystans_afir, mnoznik_afir = oblicz_luke_afir(df)
    df["dystans_do_hubu_afir_km"] = dystans_afir
    df["afir_luka_mnoznik"] = mnoznik_afir

    # Ruch dostawczy
    df["dostawcze_mnoznik"] = 1.0
    if kor.sum() > 0:
        ranga_dostawcze = df.loc[kor, "nearest_dk_lekkie_ciezarowe"].rank(pct=True)
        df.loc[kor, "dostawcze_mnoznik"] = (
            DOSTAWCZE_MNOZNIK_MIN
            + (DOSTAWCZE_MNOZNIK_MAX - DOSTAWCZE_MNOZNIK_MIN) * ranga_dostawcze.fillna(0.5)
        )

    # Korekta za konkurencję
    df["konkurencja_korekta_mnoznik"] = 1.0

    if kor.sum() > 0:
        moc_2km_kor = df.loc[kor, "existing_eipa_power_kw_active_2km"].fillna(0)
        df.loc[kor, "konkurencja_korekta_mnoznik"] = 1.0 / (1.0 + ALFA_KONKURENCJA * np.sqrt(moc_2km_kor))

    dest = df["segment"] == "docelowa"
    df["ruch_mnoznik"] = 1.0

    if dest.sum() > 0:
        ranga_ruchu = df.loc[dest, "traffic_primary_sam_osobowe"].rank(pct=True)
        df.loc[dest, "ruch_mnoznik"] = (
            RUCH_MNOZNIK_MIN + (RUCH_MNOZNIK_MAX - RUCH_MNOZNIK_MIN) * ranga_ruchu
        )

        moc_1km = df.loc[dest, "existing_eipa_power_kw_active_1km"].fillna(0)
        moc_2km = df.loc[dest, "existing_eipa_power_kw_active_2km"].fillna(0)
        moc_pierscien = (moc_2km - moc_1km).clip(lower=0)
        moc_laczna = moc_1km + moc_pierscien * WAGA_KONKURENCJI_PIERSCIEN

        df.loc[dest, "konkurencja_korekta_mnoznik"] = 1.0 / (1.0 + ALFA_KONKURENCJA * np.sqrt(moc_laczna))

    # Obliczenia populacji i energii
    if "udzial_populacji" in df.columns:
        df["udzial_populacji"] = df["udzial_populacji"].replace(0, np.nan)
        min_pop_per_powiat = df.groupby("_klucz_powiatu")["udzial_populacji"].transform("min")
        df["udzial_populacji"] = df["udzial_populacji"].fillna(min_pop_per_powiat).fillna(0.00001)

        energia_calkowita_bev_powiat = df["powiat_liczba_bev"] * PRZEBIEG_KM * ZUZYCIE_KWH_100KM / 100
        energia_calkowita_phev_powiat = (
            df["powiat_liczba_phev"] * PHEV_KOREKTA
            * PRZEBIEG_KM * ZUZYCIE_KWH_100KM / 100 * PHEV_WSPOLCZYNNIK
        )
        energia_publiczna_powiat = (
            (energia_calkowita_bev_powiat + energia_calkowita_phev_powiat)
            * df["sklonnosc_ladowania_publicznego"]
        )
        df.loc[dest, "energia_kwh_rocznie_szacunek"] = (
            energia_publiczna_powiat.loc[dest]
            * df.loc[dest, "udzial_populacji"]
            * df.loc[dest, "mnoznik_luki_infrastrukturalnej"]
        )

    df.loc[dest, "sesje_rocznie_szacunek"] = (
        df.loc[dest, "energia_kwh_rocznie_szacunek"] / SREDNIA_ENERGIA_SESJI_DOCELOWA
    )

    df["wynik_scoringowy"] = (
        df["energia_kwh_rocznie_szacunek"]
        * df["pewnosc_danych_mnoznik"]
        * df["afir_luka_mnoznik"]
        * df["ruch_mnoznik"]
        * df["konkurencja_korekta_mnoznik"]
        * df["dostawcze_mnoznik"]
    )

    for seg in ["korytarzowa", "docelowa"]:
        maska = df["segment"] == seg
        df.loc[maska, "ranking_scoringowy_procentyl"] = df.loc[maska, "wynik_scoringowy"].rank(pct=True)

    # -------------------------------------------------------------------------
    # NOVOŚĆ: FILTRACJA PRE-NMS - BUFOR 100M OD ISTNIEJĄCYCH STACJI EIPA
    # -------------------------------------------------------------------------
    print(f"Wykluczanie kandydatów leżących bliżej niż {int(BUFOR_WYKLUCZENIA_EIPA_KM*1000)}m od istniejących EIPA...")
    eipa_aktualne_mask = df["source_layer"] == "eipa_station"
    eipa_lats = df.loc[eipa_aktualne_mask, "latitude"].values
    eipa_lons = df.loc[eipa_aktualne_mask, "longitude"].values

    kandydaci_mask = df["source_layer"].isin(["fuel_station", "junction", "mop", "supermarket"])
    kandydaci_indices = df[kandydaci_mask].index

    df["za_blisko_eipa_100m"] = False

    for idx in kandydaci_indices:
        lat, lon = df.at[idx, "latitude"], df.at[idx, "longitude"]
        dists = haversine_km(lat, lon, eipa_lats, eipa_lons)
        if (dists < BUFOR_WYKLUCZENIA_EIPA_KM).any():
            df.at[idx, "za_blisko_eipa_100m"] = True

    wykluczone_cnt = df["za_blisko_eipa_100m"].sum()
    print(f"Wykluczono {wykluczone_cnt} kandydatów znajdujących się na istniejących stacjach EIPA.")

    # -------------------------------------------------------------------------
    # NMS Top 1000 z promieniem 2.0 km dla czystych kandydatów
    # -------------------------------------------------------------------------
    print(f"Oznaczanie Top {TOP_N_PUNKTOW} NMS (promień {PROMIEN_NMS_KM} km) dla nowych kandydatów...")

    df["czy_top1000_nms"] = False
    df["pozycja_ranking_nms"] = np.nan

    for seg in ["korytarzowa", "docelowa"]:
        maska_seg = (df["segment"] == seg) & kandydaci_mask & (~df["za_blisko_eipa_100m"])
        sub_df = df[maska_seg]
        
        nms_map = oznacz_top_n_nms(
            sub_df, 
            top_n=TOP_N_PUNKTOW, 
            radius_km=PROMIEN_NMS_KM, 
            score_col="wynik_scoringowy", 
            lat_col="latitude", 
            lon_col="longitude"
        )
        
        df.loc[sub_df.index, "pozycja_ranking_nms"] = sub_df.index.map(nms_map)
        df.loc[sub_df.index, "czy_top1000_nms"] = df.loc[sub_df.index, "pozycja_ranking_nms"].notna()

    # Składowe wyniki
    df["skladowa_udzial_konkurencji"] = df.get("udzial_lokalizacji", pd.NA)
    df["skladowa_pewnosc_danych"] = df["pewnosc_danych_mnoznik"]
    df["skladowa_luka_afir"] = df["afir_luka_mnoznik"]
    df["skladowa_ruch_drogowy"] = df["ruch_mnoznik"]
    df["skladowa_ruch_dostawczy"] = df["dostawcze_mnoznik"]
    df["skladowa_korekta_konkurencji"] = df["konkurencja_korekta_mnoznik"]
    df["skladowa_luka_infrastrukturalna"] = df.get("mnoznik_luki_infrastrukturalnej", pd.NA)
    df["skladowa_sklonnosc_publiczna"] = df.get("sklonnosc_ladowania_publicznego", pd.NA)

    df.to_csv(PLIK_WYJSCIOWY, index=False)
    print(f"\nZapisano {PLIK_WYJSCIOWY}, kształt: {df.shape}")

    return df

if __name__ == "__main__":
    df = main()

Wczytano 23921 wierszy z pliku wejsciowego.
Po deduplikacji stacji EIPA liczba wszystkich wierszy: 22734
Uzupełniono moc dla 237 stacji EIPA wartością mediany: 46.0 kW
Licze odleglosc do najblizszego hubu AFIR...
Huby spelniajace prog AFIR (>=400kW): 219
Wykluczanie kandydatów leżących bliżej niż 100m od istniejących EIPA...
Wykluczono 1910 kandydatów znajdujących się na istniejących stacjach EIPA.
Oznaczanie Top 1000 NMS (promień 1.0 km) dla nowych kandydatów...

Zapisano ../data/candidate_locations_ze_scoringiem.csv, kształt: (22734, 105)
